# Querying the lake: temporal analytics on AIS trajectories

The [quickstart](./mobility-lakehouse-quickstart.ipynb) wrote a TemporalParquet
lake and pruned it. This notebook shows **what temporal types give you that plain
points cannot** — read straight from the lake, no re-encoding:

1. per-trajectory metrics (length, average speed, duration),
2. **time travel** — where was each vessel at a given instant,
3. a temporal slice — the trajectory during a time window,
4. a spatiotemporal query — vessels in a region during a window, pruned by the
   covering columns.

Every cell runs against a real MobilityDuck engine (see the quickstart's *Setup*).

In [1]:
import os, json, subprocess, pandas as pd

MOBILITYDUCK = os.environ.get("MOBILITYDUCK", "duckdb")
DB = "lake_analytics.duckdb"

def duck(sql: str) -> pd.DataFrame:
    r = subprocess.run([MOBILITYDUCK, DB, "-json", "-c", sql],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr.strip())
    return pd.DataFrame(json.loads(r.stdout or "[]"))

## Build a small lake

A handful of vessels, each a `tgeogpoint` trajectory, written to a TemporalParquet
shard with covering columns (see the quickstart for the details).

In [2]:
duck("""
CREATE OR REPLACE TABLE raw(mmsi BIGINT, ts TIMESTAMPTZ, lon DOUBLE, lat DOUBLE);
INSERT INTO raw VALUES
  (211512000,'2026-02-26 08:00:00+00',4.40,51.20),(211512000,'2026-02-26 08:45:00+00',4.55,51.27),
  (211512000,'2026-02-26 09:30:00+00',4.72,51.36),(211512000,'2026-02-26 10:30:00+00',4.95,51.48),
  (244660000,'2026-02-26 08:15:00+00',3.10,51.95),(244660000,'2026-02-26 09:10:00+00',3.48,52.00),
  (244660000,'2026-02-26 10:05:00+00',3.80,52.10),(244660000,'2026-02-26 11:00:00+00',4.05,52.20),
  (305000000,'2026-02-26 08:30:00+00',4.30,51.40),(305000000,'2026-02-26 09:15:00+00',4.58,51.44),
  (305000000,'2026-02-26 10:00:00+00',4.83,51.52),(305000000,'2026-02-26 10:50:00+00',5.02,51.58);
""")

duck("""
COPY (
  SELECT mmsi, asBinary(traj) AS traj,
         Xmin(stbox(traj)) xmin, Xmax(stbox(traj)) xmax,
         Ymin(stbox(traj)) ymin, Ymax(stbox(traj)) ymax,
         Tmin(stbox(traj)) tmin, Tmax(stbox(traj)) tmax, SRID(traj) srid
  FROM (SELECT mmsi, tgeogpointSeq(list(TGEOGPOINT(ST_Point(lon,lat), ts) ORDER BY ts)) traj
        FROM raw GROUP BY mmsi)
) TO 'analytics_shard.parquet' (FORMAT PARQUET);
""")
duck("SELECT count(*) AS vessels FROM read_parquet('analytics_shard.parquet')")

,vessels
0,3


## 1. Per-trajectory metrics

`length` (geodetic → metres), `speed` (a `tfloat` → time-weighted average), and
`duration` come straight from the temporal value. None of this is expressible on a
table of isolated points.

In [3]:
duck("""
WITH t AS (SELECT mmsi, tgeogpointFromBinary(traj) AS traj
           FROM read_parquet('analytics_shard.parquet'))
SELECT mmsi,
       round(length(traj)/1000, 1)        AS length_km,
       round(twAvg(speed(traj))*3.6, 1)    AS avg_kmh,
       duration(traj)                      AS duration
FROM t ORDER BY mmsi
""")

,mmsi,length_km,avg_kmh,duration
0,211512000,49.4,19.8,02:30:00
1,244660000,71.7,26.1,02:45:00
2,305000000,54.3,23.3,02:20:00


## 2. Time travel: where was each vessel at 09:00?

`valueAtTimestamp` evaluates the trajectory function at any instant — interpolating
between pings. This is *within-trajectory* time travel (distinct from Iceberg
snapshot time travel, which moves between table versions).

In [4]:
duck("""
WITH t AS (SELECT mmsi, tgeogpointFromBinary(traj) AS traj
           FROM read_parquet('analytics_shard.parquet'))
SELECT mmsi,
       ST_AsText(valueAtTimestamp(traj, TIMESTAMPTZ '2026-02-26 09:00:00+00')) AS position_at_0900
FROM t ORDER BY mmsi
""")

,mmsi,position_at_0900
0,211512000,POINT (4.606592575078476 51.30002733329265)
1,244660000,POINT (3.410846143776197 51.99100009393464)
2,305000000,POINT (4.486612252951504 51.42674079769224)


## 3. A temporal slice

`atTime` restricts a trajectory to a time window, interpolating the endpoints —
the sub-trajectory actually travelled during the window.

In [5]:
duck("""
WITH t AS (SELECT mmsi, tgeogpointFromBinary(traj) AS traj
           FROM read_parquet('analytics_shard.parquet'))
SELECT mmsi,
       numInstants(atTime(traj, tstzspan '[2026-02-26 09:00:00+00, 2026-02-26 10:00:00+00]')) AS pings_in_window,
       round(length(atTime(traj, tstzspan '[2026-02-26 09:00:00+00, 2026-02-26 10:00:00+00]'))/1000, 1) AS km_in_window
FROM t ORDER BY mmsi
""")

,mmsi,pings_in_window,km_in_window
0,211512000,3,20.8
1,244660000,3,27.2
2,305000000,3,26.2


## 4. Spatiotemporal query: who was in the box during the window?

The lakehouse-native filter: the covering columns prune by bounding box and time
before any trajectory is decoded, then a temporal predicate confirms the slice is
non-empty. Here: vessels whose path's bounds meet the Scheldt-approaches box during
the 09:00–10:30 window.

In [6]:
duck("""
SELECT mmsi
FROM read_parquet('analytics_shard.parquet')
WHERE tmax >= TIMESTAMPTZ '2026-02-26 09:00:00+00'        -- temporal overlap
  AND tmin <  TIMESTAMPTZ '2026-02-26 10:30:00+00'
  AND xmax >= 4.5 AND xmin <= 5.1                          -- covering-column space prune
  AND ymax >= 51.3 AND ymin <= 51.7
ORDER BY mmsi
""")

,mmsi
0,211512000
1,305000000


## What this shows

The lake stores trajectories as open, lossless TemporalParquet — yet you query
them as **functions of time**: length and speed, position at any instant, slices
over windows, and spatiotemporal filters pruned by the covering columns. The same
data, the same queries, run on MobilityDB, MobilityDuck, and MobilitySpark.

Next: the [format specification](../spec/temporalparquet.md), the
[covering-columns](../spec/covering-columns.md) mechanism, and the live
[AIS Iceberg Explorer](https://ais-explorer-833836401560.europe-west1.run.app/).